In [0]:
# =============================================================================
# NOTEBOOK  : silver_pos_transactions_stream
# PURPOSE   : bronze.pos_transactions (stream rows) → silver.pos_transactions
# LAYER     : Silver (clean, type-cast, dedup, merge)
# FREQUENCY : Continuous micro-batch (JOB A — runs 24/7)
# TRIGGER   : availableNow (dev) / no trigger (prod)
#
# MERGE & DEDUP LOGIC:
#   Source  : bronze.pos_transactions WHERE _source = 'pos_realtime_stream'
#   Checkpoint tracks bronze Delta log offset — not row-level filters
#   Stream and batch silver have SEPARATE checkpoints — no interference
#
#   Within micro-batch dedup:
#     Assumption: duplicate records in same micro-batch are EXACTLY identical
#     → deduplicate on transaction_id BEFORE merge via dropDuplicates
#     WHY before merge: Delta MERGE does not deduplicate source df internally
#     Two rows with same key in source = two inserts regardless of merge logic
#
#   MERGE cases:
#     Case 1: transaction_id NOT in silver → INSERT
#     Case 2: transaction_id IN silver     → IGNORE (no whenMatchedUpdate)
#
#   Stream silver NEVER updates or deletes silver records
#   Updates and deletes are exclusively batch silver's responsibility
# =============================================================================
 
# ── Imports & Config ──────────────────────────────────────────────────
import sys, json
 
_notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
PROJECT_ROOT = "/Workspace" + _notebook_path.split("/mini_project_a3t7/")[0] + "/mini_project_a3t7"
 
sys.path.insert(0, PROJECT_ROOT)
 
from utils.audit import start_audit, end_audit
from utils.schema_registry import SILVER_POS_TRANSACTIONS_SCHEMA
 
from pyspark.sql.functions import (
    current_timestamp, lit, col,
    to_date, to_timestamp
)
from pyspark.sql.types import DecimalType
from delta.tables import DeltaTable
 
with open(f"{PROJECT_ROOT}/config/config.json") as f:
    cfg = json.load(f)
 
SOURCE_TABLE = cfg["tables"]["bronze_pos_transactions"]
TARGET_TABLE = cfg["tables"]["silver_pos_transactions"]
CHECKPOINT   = cfg["checkpoint_paths"]["silver_pos_transactions_stream"]
PIPELINE     = "silver_pos_transactions_stream"

In [0]:
# ── foreachBatch function + Stream ────────────────────────────────────
 
def process_microbatch(micro_batch_df, microbatch_id):
    """
    Called by Spark for every micro-batch of bronze rows.
 
    FILTER   : Only stream rows (_source = pos_realtime_stream)
    DEDUP    : dropDuplicates on transaction_id before MERGE
               Assumption: duplicates are exactly identical records
    MERGE    : Case 1 → INSERT, Case 2 → IGNORE
    """
 
    # ── executor-side imports ─────────────────────────────────────────────────
    import sys
    sys.path.insert(0, PROJECT_ROOT)
    from utils.audit import start_audit, end_audit
    from utils.schema_registry import SILVER_POS_TRANSACTIONS_SCHEMA
 
    # ── FILTER — stream rows only ─────────────────────────────────────────────
    micro_batch_df = micro_batch_df.filter(
        col("_source") == "pos_realtime_stream"
    )
 
    if micro_batch_df.isEmpty():
        return
 
    run_id = start_audit(spark, PROJECT_ROOT, PIPELINE, TARGET_TABLE, SOURCE_TABLE)
 
    try:
        rows_read = micro_batch_df.count()
 
        # ── TRANSFORM ─────────────────────────────────────────────────────────
        df = (
            micro_batch_df
 
            # Cast timestamp ISO string → TimestampType
            # Renamed transaction_ts — avoids Spark reserved word 'timestamp'
            .withColumn("transaction_ts",
                to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'"))
 
            # Money columns Double → Decimal(10,2) for precision
            .withColumn("unit_price",
                col("unit_price").cast(DecimalType(10, 2)))
            .withColumn("total_amount",
                col("total_amount").cast(DecimalType(10, 2)))
 
            # transaction_date from event time — partition column
            # Use transaction_ts (when event happened) not ingested_at
            .withColumn("transaction_date",
                to_date(
                    to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss'Z'")
                ).cast("string"))
 
            # Silver audit columns
            .withColumn("processed_at",    current_timestamp())
            .withColumn("pipeline_run_id", lit(run_id))
            .withColumn("source_system",   lit("pos_realtime_stream"))
 
            # Enforce silver schema — drops all bronze-only columns
            .select([f.name for f in SILVER_POS_TRANSACTIONS_SCHEMA.fields])
        )
 
        # ── DEDUP WITHIN MICRO-BATCH ──────────────────────────────────────────
        # Must happen BEFORE merge — Delta MERGE does not dedup source df
        # Two rows with same transaction_id = two inserts
        df = df.dropDuplicates(["transaction_id"])
 
        rows_after_dedup = df.count()
        if rows_read != rows_after_dedup:
            print(f"[DEDUP] dropped {rows_read - rows_after_dedup} "
                  f"duplicate stream rows in microbatch_id={microbatch_id}")
 
        # ── MERGE INTO SILVER ──────────────────────────────────────────────────
        # Case 1: not in silver → INSERT
        # Case 2: in silver     → no whenMatchedUpdate → IGNORE
        (
            DeltaTable.forName(spark, TARGET_TABLE).alias("t")
            .merge(
                df.alias("s"),
                "t.transaction_id = s.transaction_id"
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
 
        metrics = (
            spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1")
            .select("operationMetrics")
            .collect()[0][0]
        )
        rows_inserted = int(metrics.get("numTargetRowsInserted", 0))
 
        end_audit(
            spark, PROJECT_ROOT, run_id, "SUCCESS",
            rows_read=rows_read,
            rows_written=rows_inserted,
            extra={
                "rows_inserted": str(rows_inserted),
                "microbatch_id": str(microbatch_id)
            }
        )
 
    except Exception as e:
        end_audit(spark, PROJECT_ROOT, run_id, "FAILED", error=str(e))
        raise

# ── READ BRONZE AS STREAM ─────────────────────────────────────────────────────
bronze_stream = (
    spark.readStream
    .format("delta")
    .option("ignoreChanges", "true")
    .table(SOURCE_TABLE)
)
 
# ── WRITE WITH foreachBatch ───────────────────────────────────────────────────
# trigger(availableNow=True) for dev
# Remove trigger entirely for prod JOB A — continuous micro-batch 24/7
query = (
    bronze_stream.writeStream
    .foreachBatch(process_microbatch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .start()
)
 
query.awaitTermination()

In [0]:
# ── Verify last run ───────────────────────────────────────────────────
from pyspark.sql.functions import col
 
# 1. Audit status
spark.read.table("azure3_team7_project.platform.pipeline_audit") \
    .filter(col("pipeline_name") == PIPELINE) \
    .orderBy(col("start_time").desc()) \
    .limit(1) \
    .select("status", "rows_read", "rows_written", "extra_metadata") \
    .show(truncate=False)
 
# 2. Silver row count
print("Silver row count:", spark.read.table(TARGET_TABLE).count())
 
# 3. Nulls in key columns
print("Null transaction_ids:",
    spark.read.table(TARGET_TABLE)
    .filter(col("transaction_id").isNull())
    .count()
)
 
# 4. Delta history
spark.sql(f"DESCRIBE HISTORY {TARGET_TABLE} LIMIT 1") \
    .select("operation", "operationMetrics") \
    .show(truncate=False)